# Chroma DB 실습

In [1]:
from langchain_chroma import Chroma
from langchain.retrievers import EnsembleRetriever
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader

In [2]:
embedding = OpenAIEmbeddings(model="text-embedding-3-small")
len(embedding.embed_query("hello world"))

1536

In [3]:
sv_2024_pdf_path = "../../data/Sustainability_report_2024_kr.pdf"

loader = PyPDFLoader(sv_2024_pdf_path)
docs_2024 = loader.load()

len(docs_2024)

83

In [4]:
sv_2025_pdf_path = "../../data/Samsung_Electronics_Sustainability_Report_2025_KOR.pdf"

loader = PyPDFLoader(sv_2025_pdf_path)
docs_2025 = loader.load()

len(docs_2025)

87

In [5]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
)

chunk_2024 = splitter.split_documents(docs_2024)
chunk_2025 = splitter.split_documents(docs_2025)

In [ ]:
# 벡터 스토어 생성
vs_2024 = Chroma.from_documents(
    documents=chunk_2024,
    collection_name="samsung_2024", # db명
    persist_directory="./chroma_store",
    embedding=embedding
)

vs_2025 = Chroma.from_documents(
    documents=chunk_2025,
    collection_name="samsung_2025",
    persist_directory="./chroma_store",
    embedding=embedding
)

In [7]:
r_2024 = vs_2024.as_retriever(search_kwargs={"k": 5})
r_2025 = vs_2025.as_retriever(search_kwargs={"k": 5})

In [8]:
r_2024.invoke("2024년 삼성 경영 관련 내용 알려줘")

[Document(id='0f85d796-2e0c-428a-a6c4-ed016e27f43b', metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign 15.1 (Macintosh)', 'moddate': '2024-11-25T11:10:46+09:00', 'page': 29, 'page_label': '30', 'creationdate': '2024-11-25T11:10:32+09:00', 'trapped': '/False', 'total_pages': 83, 'source': '../../data/Sustainability_report_2024_kr.pdf'}, page_content='삼성전자 지속가능경영보고서 2024 30Our Company AppendixMateriality Assessment Facts & Figures PrinciplePlanet People\n사회적 책임을 다하며 미래로 함께 나아갑니다.\nPeople\n31    임직원\n39    공급망\n45    사회공헌\n48    개인정보보호/보안\n50    고객의 안전/품질'),
 Document(id='8cfed03e-b739-4a5c-ab68-5f3ec2d42aaa', metadata={'total_pages': 83, 'moddate': '2024-11-25T11:10:46+09:00', 'source': '../../data/Sustainability_report_2024_kr.pdf', 'page': 81, 'creationdate': '2024-11-25T11:10:32+09:00', 'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign 15.1 (Macintosh)', 'page_label': '82', 'trapped': '/False'}, page_content='삼성전자 지속가능경영보고서 2024\n82\nOur Company

In [9]:
r_2025.invoke("2025년 삼성 경영 관련 내용 알려줘")

[Document(id='e063d489-7e36-40d3-9883-66552a81633b', metadata={'source': '../../data/Samsung_Electronics_Sustainability_Report_2025_KOR.pdf', 'creator': 'Adobe InDesign 15.1 (Macintosh)', 'page_label': '5', 'total_pages': 87, 'trapped': '/False', 'page': 4, 'moddate': '2025-09-04T16:51:11+09:00', 'producer': 'Adobe PDF Library 15.0', 'creationdate': '2025-07-10T16:11:16+09:00'}, page_content='삼성전자 지속가능경영보고서 2025\n05\nOur Company AppendixFacts & Figures PrinciplePlanet People\n회사소개\nAbout Us\n삼성전자주식회사(이하 삼성전자)는 인재와 기술을 기반으로 최고의 제품과 서비스를 창출함으로써 인류사회에 기여하는 글로벌 초일류 기업을 지향합니다. 이를 위해 삼성전자가 지켜나갈 약속인 5가지 경영원칙 을 세부원칙과 \n행동지침으로 구체화하고, 삼성전자 임직원이 지켜야 할 행동규범 으로 제정하여 모든 경영활동의 기준으로 삼고 있습니다. 앞으로도 삼성전자는 조직문화에 5가지 핵심가치 를 내재화하여 지속적으로 성장해갈 것입니다.\n사업부문 및 글로벌 네트워크 소개\n삼성전자는 제품 특성에 따라 DX(Device eXperience)와 DS(Device Solutions) 2개 부문으로 나뉘어 독립적으로 운영되고 있습니다. DX부문은 스마트폰, 네트워크 시스템, 컴퓨터, TV, 냉장고, 세탁기, 에어컨, 의료기기 등 완제품을 \n생산·판매하고 있으며, DS부문은 메모리 사업, Foundry 사업, System LSI 사업으로 구성되어 DRAM, NAND Flash, 모바일AP 등의 반도체 부품을

In [10]:
ensemble_ret = EnsembleRetriever(retrievers=[r_2024, r_2025], weights=[0.5, 0.5])
result = ensemble_ret.invoke("2024년과 2025의 삼성 차이 요약해줘")
print(result[0].page_content)

삼성전자 지속가능경영보고서 2024
04
Our Company AppendixMateriality Assessment Facts & Figures PrinciplePlanet People
CEO 메시지
Message from 
Our CEO
주주, 고객, 협력회사, 그리고 임직원 여러분,
2023년은 고금리와 인플레이션, 지정학적 이슈 등 매우 불확실한 
거시경제 환경과 함께, 메모리 산업 부진과 다양한 제품군에서의 경쟁 
심화로 삼성전자에게 매우 어려운 한 해였습니다. 이토록 대내외적으로 
어려운 환경에서도 지속 성장의 기반 마련을 위해 역대 최고 수준의 28.3
조원을 연구개발에 투자하고, 53.1조원 수준의 전략적 시설투자로 기술 
리더십을 강화하며 중장기 수요에 미리 대응할 수 있었던 것은 삼성전자를 
아껴주시는 이해관계자 여러분의 관심과 격려 덕분입니다. 다시 한 번 
깊이 감사 드립니다.
급격한 변화를 겪고 있는 경제 상황에 맞춰, 기업의 지속가능경영 
분야에서도 많은 변화가 일어나고 있습니다. 특히 기업의 지속가능경영 
활동 정보 공개는 글로벌 비재무정보 공시 제도의 확산에 맞춰, 새로운 
국면을 맞고 있습니다. 국제회계기준재단(IFRS Foundation)이 2023년 
6월 지속가능성 지표를 확정한 것을 시작으로, EU의 지속가능성 보고지침
(CSRD)과 미국 증권거래위원회(SEC) 기후공시 기준 역시 세부 내용을 
순차적으로 확정하며 ESG 정보의 의무 공시 시대가 열리고 있습니다.
이와 함께, EU 탄소국경조정제도(CBAM)와 EU 배터리규제 등을 통한 
환경규제 역시 지속 강화되는 추세이고, 독일에서는 공급망의 인권과 
근로환경 관리를 의무화하는 공급망실사법이 2023년 발효되었으며, 
2024년 5월 EU 공급망 실사지침(CSDDD)이 확정되는 등 인권 분야에 
대한 관심 또한 지속 고조되고 있습니다. 
삼성전자는 이러한 추세에 맞춰 지속가능한 미래를 위한 노력을 계속해


In [11]:
import chromadb
from chromadb.config import Settings

client = chromadb.PersistentClient(path="./chroma_store")

src24 = client.get_collection("samsung_2024")
src25 = client.get_collection("samsung_2025")

In [12]:
# 새롭게 전체 데이터베이스 생성 (빈 공간 생성)
vs_all = Chroma( 
    collection_name="samsung_all",
    persist_directory="./chroma_store",
    embedding_function=embedding
)

In [13]:
dst = client.get_collection("samsung_all")

In [14]:
# 2. samsung_2024 → samsung_all 복사
batch = 1000
offset = 0
while True:
    batch_data = src24.get(
        include=["embeddings", "metadatas", "documents"],
        limit=batch, offset=offset
    )
    ids = batch_data["ids"]
    if not ids:
        break

    # ID 충돌 방지: 접두사 붙이기
    new_ids = [f"{src24.name}::{i}" for i in ids]

    dst.upsert(
        ids=new_ids,
        documents=batch_data["documents"],
        embeddings=batch_data["embeddings"],   # 이미 계산된 벡터 재사용
        metadatas=batch_data["metadatas"],
    )
    offset += len(ids)
    print(f"[{src24.name}] moved {offset} docs so far...")

[samsung_2024] moved 207 docs so far...


In [15]:
# 3. samsung_2025 → samsung_all 복사
batch = 1000
offset = 0
while True:
    batch_data = src25.get(
        include=["embeddings", "metadatas", "documents"],
        limit=batch, offset=offset
    )
    ids = batch_data["ids"]
    if not ids:
        break

    new_ids = [f"{src25.name}::{i}" for i in ids]

    dst.upsert(
        ids=new_ids,
        documents=batch_data["documents"],
        embeddings=batch_data["embeddings"],
        metadatas=batch_data["metadatas"],
    )
    offset += len(ids)
    print(f"[{src25.name}] moved {offset} docs so far...")

[samsung_2025] moved 237 docs so far...


In [16]:
vs_all._collection.count()

444

In [17]:
vs_2024._collection.count()

207

In [18]:
vs_all.similarity_search("2024년과 2025년의 변화 요약", k=5)

[Document(id='samsung_2025::3006145d-709e-468a-9a7e-28a32a94f3bd', metadata={'creator': 'Adobe InDesign 15.1 (Macintosh)', 'total_pages': 87, 'trapped': '/False', 'creationdate': '2025-07-10T16:11:16+09:00', 'page_label': '11', 'producer': 'Adobe PDF Library 15.0', 'source': '../../data/Samsung_Electronics_Sustainability_Report_2025_KOR.pdf', 'moddate': '2025-09-04T16:51:11+09:00', 'page': 10}, page_content="삼성전자 지속가능경영보고서 2025\n11\nDX부문\n추진체계와 주요성과\nGovernance and Major Progress\n추진체계\n삼성전자 이사회는 회사의 최 고 의사결정기구로, 이사회 산하  \n지속가능경영위원회가 기후변화, 자원순환 등 환경 관련 전략과 \n목표를 승인하고, 주요 활동을 관리 ·감독하고 있습니다. 2022년 \n지속가능경영위원회는 중장기 기후변화 대응, 자원순환 목표를 아우르는 \n'新환경경영전략 '을 결의하였으며, 2023년부터 매년 주요 성과를 \n위원회 안건에 포함하여 검토하고 있습니다.\nDX부문장은 환경경영전략 수립, 이 행 과제 발 굴, 투자 실시 등 주요 \n사안에 대한 책 임과 권한을 가지며, 각  사업부장, 관련 부서장 들과 \n함께 DX부문 지속가능경영협의회를 운영하고 있습니다. 2024년 \n지속가능경영협의회는 연 3회 진행되었으며, 사업부 및 유관부서 주요 \n과제의 연간 계획과 진척 사항이 공유되었습니다. \n환경경영 계획의 실행은 지속가능경영추진센터, Global EHS실, 지역별 \n환경 전 담조직, 사업부 지속가능경영사 무국 등이 담당 하며, 원활한 \n논의를 위해 관련 부서 